# 01 — Preprocessing & Quality Control

This notebook covers:
- Loading raw scRNA-seq data
- Computing QC metrics (genes per cell, UMI counts, % mitochondrial)
- Visualizing QC distributions
- Filtering low-quality cells and genes
- Doublet detection (optional, requires Scrublet)


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config as cfg
from utils import compute_qc_metrics, filter_cells_and_genes, detect_doublets
from utils import plot_qc_metrics, plot_qc_scatter, plot_highest_expressed_genes
from utils import load_data, save_adata, ensure_dirs

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor='white')
%matplotlib inline

ensure_dirs(cfg.RESULTS_DIR, cfg.FIGURES_DIR)
print('Setup complete ✓')

## 1. Load Data

In [ ]:
cfg_dict = {k: getattr(cfg, k) for k in dir(cfg) if not k.startswith('_')}
adata = load_data(cfg_dict)
adata.var_names_make_unique()
print(adata)

## 2. Compute QC Metrics

In [ ]:
adata = compute_qc_metrics(adata, mito_prefix=cfg.QC['mito_prefix'])
adata.obs[['n_genes_by_counts','total_counts','pct_counts_mt']].describe()

## 3. Visualise QC (Before Filtering)

In [ ]:
plot_qc_metrics(adata)

In [ ]:
plot_qc_scatter(adata)

In [ ]:
plot_highest_expressed_genes(adata, n_top=20)

## 4. Choose Thresholds

Inspect the plots above and adjust thresholds in `config.py` → `QC` dict.

In [ ]:
print('Current QC thresholds:')
for k, v in cfg.QC.items():
    print(f'  {k}: {v}')

## 5. Filter Cells & Genes

In [ ]:
adata = filter_cells_and_genes(adata, cfg.QC)
print(f'After filtering: {adata.n_obs} cells × {adata.n_vars} genes')

In [ ]:
plot_qc_metrics(adata)
plot_qc_scatter(adata)

## 6. (Optional) Doublet Detection

In [ ]:
adata = detect_doublets(adata)
if 'predicted_doublet' in adata.obs.columns:
    print(adata.obs['predicted_doublet'].value_counts())
    adata = adata[~adata.obs['predicted_doublet']].copy()
    print(f'After doublet removal: {adata.n_obs} cells')

## 7. Save

In [ ]:
save_adata(adata, cfg.ADATA_PROCESSED_PATH)
print('Done ✓')